# Step 1: Import necessary imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from functools import reduce
from operator import or_
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Step 1.1: Ensure we are using correct catalog and schema

In [0]:
%sql

USE CATALOG global_stock_market;
USE SCHEMA bronze;

SELECT current_catalog(), current_schema()

### Remove all rows in invalid_silver_tables for demo purposes

In [0]:
%sql
DROP SCHEMA IF EXISTS  global_stock_market.invalid_silver_tables CASCADE;
CREATE SCHEMA IF NOT EXISTS global_stock_market.invalid_silver_tables;

DROP SCHEMA IF EXISTS  global_stock_market.silver CASCADE;
CREATE SCHEMA IF NOT EXISTS global_stock_market.silver;

# Step 2: Get all delta tables in bronze schema

In [0]:
tables = spark.catalog.listTables("bronze")

tables_list = []

for table in tables:
    tables_list.append(table.name)

print(tables_list)

# Step 3: Read all tables into dictionary

In [0]:
tables_dict = {}

for table_name in tables_list:
    tables_dict[table_name] = spark.read.table(tableName=table_name)

# verify turkey stock market dataframe

tables_dict["bist100_turkey"].show(3)

# Step 4: Data Transformation

### Data quality checks for each table:

- Correct data types
- Trim string columns
- Ensure no nulls in Ticker or Date or Price columns
- make sure volume is not negative
- Ensure no prices columns are negative

*If any rows do not conform to the data checks, it will be filtered out first but kept at invalid_silver.invalid_rows schema for investigation*

### 4.1 Inspect tables data types

In [0]:
# I have noticed that some tables have volumn column as "int"
# whereas others have "bigint"
# I will change all of them to "bigint" as wide casting is fine

for table_name, df in tables_dict.items():
    df = df.withColumn("Volume", df["Volume"].cast("bigint"))
    tables_dict[table_name] = df

# quick verification check of data types of table
for table_name, df in tables_dict.items():
    print(f"Table name: {table_name}")
    print(df.dtypes)
    print()

#### It looks our tables have correct data types !! 

### 4.2 Trim string columns in each table

In [0]:
def trimming_string_columns(table):
    df = tables_dict[table]

    return df.withColumns({
        c: F.trim(c)
        for c, t in df.dtypes
        if t == "string"
    })

In [0]:
# implement trimming string columns

for table_name in tables_dict.keys():
    tables_dict[table_name] = trimming_string_columns(table_name)

print("Done !! All String columns are trimmed")

### 4.3 Ensure important columns do not have null

In [0]:
# These columns must not have nulls
# NULL CHECK

non_null_columns = ["Ticker", "Date", "Open", "High", "Low", "Close"]

def null_check(table):

    print(f"Validating table: {table}")

    df = tables_dict[table]

    # 1. Define required columns for this table , create a local copy for each table 
    required_columns = non_null_columns.copy()

    if table == "index_levels":
        required_columns = [
            "Index", "Date", "Open", "High", "Low", "Close"
        ]

    # 2. Check that all required columns exist
    missing_columns = set(required_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{table} is missing columns: {missing_columns}"
        )

    # 3. Check whether ANY required column is NULL
    null_condition = reduce(
        or_,
        [F.col(column).isNull() for column in required_columns]
    )

    # 4. Extract invalid rows and add a reason column
    invalid_df = (
        df.filter(null_condition)
        .withColumn("reason", F.lit("contain null"))
    )

    # 5. Keep only clean rows
    clean_df = df.filter(~null_condition)

    # 6. Write invalid rows to the corresponding Delta table if at least one row exist or else skip 
    if invalid_df.limit(1).count() > 0:
        (
            invalid_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(
                f"global_stock_market.invalid_silver_tables.{table}"
            )
        )

    # 7. Replace the original DataFrame with clean rows
    tables_dict[table] = clean_df

    print(f"Valid Rows: {clean_df.count()}")
    print(f"Invalid Rows: {invalid_df.count()}")
    print()


In [0]:
# Implement null check

for table_name in tables_dict.keys():
    null_check(table_name)

### 4.4: Ensure prices columns are not null in clean dataframe

In [0]:
# Ensuring price columns are not negative 

prices_columns = ["Open", "High", "Low", "Close"]

def check_prices(table):

    print(f"Validating table: {table}")

    df = tables_dict[table]

    prices_condition = reduce(
        or_,
        [F.col(column) < 0 for column in prices_columns]
    )

    invalid_df = (
        df.filter(prices_condition)
        .withColumn("reason", F.lit("price is negative"))
    )

    clean_df = df.filter(~prices_condition)

    if invalid_df.limit(1).count() > 0:
        (
            invalid_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(f"global_stock_market.invalid_silver_tables.{table}")
        )

    tables_dict[table] = clean_df

    print(f"Valid Rows: {clean_df.count()}")
    print(f"Invalid Rows: {invalid_df.count()}")
    print()
    

In [0]:
# Implement prices check

for table_name in tables_dict.keys():
    check_prices(table_name)


### 4.5 Duplicate check

In [0]:
def duplicate_check(table):

    print(f"Validating table: {table}")
    
    # important key is ticker + date / index + date

    if table == "index_levels":
        key = ["Index", "Date"]
    else:
        key = ["Ticker", "Date"]

    df = tables_dict[table]

    duplicate_condition = df.groupBy(key).count().filter(F.col("count")> 1).select(*key)  # unpack to only select key columns

    invalid_df = (
        df.join(duplicate_condition, on=key, how="inner")
        .withColumn(
            "reason",
            F.lit("duplicate_key_records")
        )
    )

    clean_df = df.join(
        duplicate_condition,
        on=key,
        how="left_anti" # rows not satisfying the duplicate condition
    )

    # Write invalid rows to Delta table
    if invalid_df.limit(1).count() > 0:
        (
            invalid_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(
                f"global_stock_market.invalid_silver_tables.{table}"
            )
        )

    # 6. Update the dictionary with clean records
    tables_dict[table] = clean_df

    print(f"Valid Rows: {clean_df.count()}")
    print(f"Invalid Rows: {invalid_df.count()}")
    print()

In [0]:
# Implement duplicate check

for table_name in tables_dict.keys():
    duplicate_check(table_name)

### 4.6 OHLC price consistency (Open, High, Low, Close)

In [0]:
def ohlc_check(table):

    print(f"Validating table: {table}")

    df = tables_dict[table]

    invalid_condition = (
    (F.col("High") < F.col("Low")) |
    (F.col("High") < F.col("Open")) |
    (F.col("High") < F.col("Close")) |
    (F.col("Low") > F.col("Open")) |
    (F.col("Low") > F.col("Close"))
    )

    invalid_df = (
        df.filter(invalid_condition)
        .withColumn("reason", F.lit("OHLC check failed"))
    )

    clean_df = df.filter(~invalid_condition)

    if invalid_df.limit(1).count() > 0:
        (
            invalid_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(
                f"global_stock_market.invalid_silver_tables.{table}"
            )
        )

    tables_dict[table] = clean_df

    print(f"Valid Rows: {clean_df.count()}")
    print(f"Invalid Rows: {invalid_df.count()}")
    print()

In [0]:
# Implement ohlc check

for table_name in tables_dict.keys():
    ohlc_check(table_name)

### 4.7 Verify check to ensure 2 different "invalid" checks are in the invalid table

In [0]:
%sql

SELECT reason,  COUNT(*)
FROM global_stock_market.invalid_silver_tables.nikkei225_japan
GROUP BY reason;


# Step 5: Create Daily Returns for each column to compare index/tickers improvements more fairly

* If S&P500 Opening price is $1000 and ends with $1010 
* However, if nifty50_india starts with $100 and ends with $150 

* It doesnt mean S&P500 performed better , S&P500 improvement is only 1% wherease is nifty50_india is 50%

In [0]:
def daily_returns(table):

    df = tables_dict[table]

    if table == "index_levels":
        identifier = "Index"
        price_col = "Close"
    else:
        identifier = "Ticker"
        price_col = "Adj_Close"

    # this ensures the ticker/index prices do not get mixed up and date is arranged chronologically
    window_spec = Window.partitionBy(identifier).orderBy("Date")

    # F.lag() to retrieve the value from the previous row in the window.
    # .over() is to tell spark which group and date ordering to use
    df = df.withColumn(
        "Previous_Close",
        F.lag(F.col(price_col)).over(window_spec)
    )

    df = df.withColumn(
        "Daily_Returns_Percentage",
        F.when(
            F.col("Previous_Close").isNotNull()
            & (F.col("Previous_Close") != 0),
            F.round(
                (F.col(price_col) - F.col("Previous_Close"))
                / F.col("Previous_Close") * 100,
                2
            )
        ).otherwise(F.lit(None))
    )

    return df



In [0]:
# implementing daily_returns calculation

for table_name in tables_dict.keys():
    tables_dict[table_name] = daily_returns(table_name)

# Step 6: Attach the country of origin of the stock and the region to index_levels table

In [0]:
market_metadata = [
    ("SP500", "United States", "North America"),
    ("NIFTY50", "India", "Asia"),
    ("NIKKEI225", "Japan", "Asia"),
    ("BOVESPA", "Brazil", "South America"),
    ("FTSE100", "United Kingdom", "Europe"),
    ("DAX40", "Germany", "Europe"),
    ("TADAWUL", "Saudi Arabia", "Middle East"),
    ("SSE", "China", "Asia"),
    ("IDX", "Indonesia", "Asia"),
    ("BIST100", "Turkey", "Europe/Asia")
]

metadata_df = spark.createDataFrame(
    market_metadata, 
    ["Index" , "Country" , "Region"]
    )

tables_dict["index_levels"] = tables_dict["index_levels"].join(metadata_df, on = "Index", how = "left")

In [0]:
# verify the join and daily returns percetage column

rand = tables_dict["index_levels"].orderBy(F.random()).limit(5)

rand.show()

# Step 7: Write clean data into silver table

In [0]:
for table_name, df in tables_dict.items():
    df.write.format("delta").mode("overwrite").saveAsTable(f"global_stock_market.silver.{table_name}")
    print(f"Writing {table_name} to silver schema as delta table !")
    print()

print("Success !!")